In [2]:
from pathlib import Path
import pandas as pd
from LFPAnalysis import build_basic_pipeline_config, run_pipeline, build_event_locked_pipeline_config

data_directory = Path('/Volumes/T7 Shield/projects/guLab/Salman/EMU')
subjects = ['MS016', 'MS017', 'MS019',
'MS020', 'MS023', 'MS025', 'MS026', 'MS030', 'MS035', 'MS036']

# 'UI001', 'UI002','UI006', 'UI007'

for subject in subjects:
    data_path = data_directory / subject / 'neural' / 'Day2' / 'lfp_data.fif'
    electrode_path = Path(data_directory / subject / 'anat' / f'{subject}_labels.xlsx')
    if not electrode_path.exists():
        electrode_path = Path(data_directory / subject / 'anat' / f'{subject}_labels.csv')
    
    config = build_basic_pipeline_config(
        data_path,
        file_format='mne',
        reference_method='bipolar',
        electrode_path=electrode_path,
    )
    result = run_pipeline(config)
    # After bipolar re-reference, prep drops the superseded monopolar Raw to save RAM.
    print(f'Bipolar channels: {len(result.referenced.ch_names)}')
    print('First bipolar channels:', result.referenced.ch_names[:5])
    print('result.raw is None after re-reference:', result.raw is None)

    save_path = data_directory / subject / 'neural' / 'Day2' / 'lfp_data_bp.fif'
    result.referenced.save(save_path, overwrite=True)
    result.electrode_df.to_csv(
        data_directory / subject / 'anat' / f'{subject}_labels_bp.csv',
        index=False,
    )

    # load the beh file
    beh = pd.read_csv(f'/Volumes/T7 Shield/scratch/MemoryBandit/Salma/memory_df_{subject}_Day2.csv')

    cross_config = build_event_locked_pipeline_config(
    save_path,
    file_format='mne',
    event_name='cue_on',
    event_times=beh['cue_on'].tolist(),
    baseline_mode='zscore',
    baseline_event_times=beh['baseline_start_mem'].tolist(),
    baseline_window=(0.0, 0.75),
    tmin=-0.5,
    tmax=1.5,
    metadata={'reward': beh['reward'].tolist(), 'rpe': beh['rpe'].tolist()},
)
    cross = run_pipeline(cross_config)
    print('cross_event_baseline:', cross.metadata.get('cross_event_baseline'))
    print('has_baseline_epochs:', cross.metadata.get('has_baseline_epochs'))
    print(cross.baseline_summary.head())

Opening raw data file /Volumes/T7 Shield/projects/guLab/Salman/EMU/MS016/neural/Day2/lfp_data.fif...


/Users/salmanqasim/Documents/GitRepos/LFPAnalysis/LFPAnalysis/workflow.py:218: RuntimeWarning: This filename (/Volumes/T7 Shield/projects/guLab/Salman/EMU/MS016/neural/Day2/lfp_data.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  data = mne.io.read_raw_fif(path, preload=config.preload)


Isotrak not found
    Range : 0 ... 346218 =      0.000 ...   692.436 secs
Ready.
Reading 0 ... 346218  =      0.000 ...   692.436 secs...
Number of electrodes in the mne file is less than the number of electrodes in the localization file
sEEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=65, n_times=346219
    Range : 0 ... 346218 =      0.000 ...   692.436 secs
Ready.
Added the following bipolar channels:
lacas1-lacas2, lacas8-lacas9, lacas9-lacas10, lagit1-lagit2, lagit2-lagit3, lagit3-lagit4, lagit6-lagit7, lagit7-lagit8, lagit8-lagit9, lagit9-lagit10, laimm1-laimm2, laimm2-laimm3, laimm3-laimm4, laimm4-laimm5, laimm5-laimm6, laimm10-laimm11, laimm11-laimm12, laimm12-laimm13, laimm13-laimm14, lhpit1-lhpit2, lhpit2-lhpit3, lhpit6-lhpit7, lhpit7-lhpit8, lhpit8-lhpit9, lhpit9-lhpit10, lils2_1-lils1_1, lils1_1-lils2_2, lils2_2-lils1_2, lils1_2-lils2_3, lils2_3-lils1_3, lils1_3-lils2_4, lils2_4-lils1_4, lils1_4-lils2_5, lils2_5-lils1_5, lmcms1-

/var/folders/rs/5fd7r4bd4ps2x94n6d46k_t00000gn/T/ipykernel_81151/2097780215.py:30: RuntimeWarning: This filename (/Volumes/T7 Shield/projects/guLab/Salman/EMU/MS016/neural/Day2/lfp_data_bp.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  result.referenced.save(save_path, overwrite=True)
/Users/salmanqasim/Documents/GitRepos/LFPAnalysis/LFPAnalysis/workflow.py:218: RuntimeWarning: This filename (/Volumes/T7 Shield/projects/guLab/Salman/EMU/MS016/neural/Day2/lfp_data_bp.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  data = mne.io.read_raw_fif(path, preload=config.preload)


cross_event_baseline: True
has_baseline_epochs: True
   target         channel    mode  baseline_start  baseline_stop  \
0  epochs   lacas1-lacas2  zscore             0.0           0.75   
1  epochs   lacas8-lacas9  zscore             0.0           0.75   
2  epochs  lacas9-lacas10  zscore             0.0           0.75   
3  epochs   lagit1-lagit2  zscore             0.0           0.75   
4  epochs   lagit2-lagit3  zscore             0.0           0.75   

   baseline_mean  baseline_std  
0   6.514110e-06      0.000096  
1   5.514388e-08      0.000062  
2   5.604523e-07      0.000034  
3   5.708911e-06      0.000098  
4   1.567359e-05      0.000104  
Opening raw data file /Volumes/T7 Shield/projects/guLab/Salman/EMU/MS017/neural/Day2/lfp_data.fif...


/Users/salmanqasim/Documents/GitRepos/LFPAnalysis/LFPAnalysis/workflow.py:218: RuntimeWarning: This filename (/Volumes/T7 Shield/projects/guLab/Salman/EMU/MS017/neural/Day2/lfp_data.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  data = mne.io.read_raw_fif(path, preload=config.preload)


Isotrak not found
    Range : 0 ... 220218 =      0.000 ...   440.436 secs
Ready.
Reading 0 ... 220218  =      0.000 ...   440.436 secs...
Number of electrodes in the mne file is less than the number of electrodes in the localization file
sEEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=55, n_times=220219
    Range : 0 ... 220218 =      0.000 ...   440.436 secs
Ready.
Added the following bipolar channels:
lacas1-lacas2, lacas2-lacas3, lacas3-lacas4, lacas4-lacas5, lacas5-lacas6, lacas6-lacas7, lacas7-lacas8, lacas8-lacas9, lacas9-lacas10, laglt1-laglt2, laglt2-laglt3, laglt4-laglt5, laglt5-laglt6, laglt6-laglt7, laglt7-laglt8, laglt8-laglt9, lalps1-lalps2, lalps2-lalps3, lalps3-lalps4, lalps10-lalps11, lalps11-lalps12, lalps12-lalps13, lhplt1-lhplt2, lhplt2-lhplt3, lhplt7-lhplt8, lhplt8-lhplt9, lllt1-lllt2, lllt5-lllt6, lmcms1-lmcms2, lmcms2-lmcms3, lmcms3-lmcms4, lmcms6-lmcms7, lmlbr1-lmlbr2, lmlbr2-lmlbr3, lmlbr3-lmlbr4, lmolf1-lmolf2, lmo

/var/folders/rs/5fd7r4bd4ps2x94n6d46k_t00000gn/T/ipykernel_81151/2097780215.py:30: RuntimeWarning: This filename (/Volumes/T7 Shield/projects/guLab/Salman/EMU/MS017/neural/Day2/lfp_data_bp.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  result.referenced.save(save_path, overwrite=True)
/Users/salmanqasim/Documents/GitRepos/LFPAnalysis/LFPAnalysis/workflow.py:218: RuntimeWarning: This filename (/Volumes/T7 Shield/projects/guLab/Salman/EMU/MS017/neural/Day2/lfp_data_bp.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  data = mne.io.read_raw_fif(path, preload=config.preload)


ConfigurationError: baseline_epochs must have the same number of trials as the task epochs.

In [9]:
len(beh['cue_on'].tolist())

80

In [10]:
len(beh['baseline_start_mem'].tolist())

80